# Software comparison: fastCDS vs. other protein-to-genome tools

This notebook compares **fastCDS** with other available tools for mapping protein or domain coordinates to the genome. Coordinate agreement (Supplementary Table S1) is evaluated against the two Bioconductor `proteinToGenome` implementations (**ensembldb** and **GenomicFeatures**), as well as **TransVar** and the **Ensembl REST API**. Runtime and memory usage (Supplementary Table S2) are compared against ensembldb, GenomicFeatures, geneplot, and the Ensembl REST API.

All tools were evaluated using the same Ensembl 86 human dataset. Benchmarks for slower tools (ensembldb, TransVar, geneplot, and the Ensembl REST API) were run offline using the scripts in benchmarks/, as they require minutes to hours to complete.

For implementation details and notes on each comparator, see the wiki section Speed vs other tools.

## Setup

In [1]:
import os
import pandas as pd
from pathlib import Path

# Set FASTCDS_REPO to override when running outside a checkout.
_start = Path.cwd()
_env = os.environ.get("FASTCDS_REPO")
REPO = Path(_env) if _env else next(
    (p for p in [_start, *_start.parents]
     if (p / "benchmarks" / "compare_intervals.py").exists()), _start)

pd.set_option("display.max_columns", None, "display.width", 200)
print("repo:", REPO)


repo: /home/goguxor/Desktop/fastCDS/reproduce_benchmarking_figures


## 1. Per-tool agreement vs fastCDS (Supplementary Table S1)

Each tool was evaluated on the same 5,000-query validation set, with fastCDS re-indexed to the corresponding Ensembl release for a fair comparison. The dataset spans nine query categories, with `cds_incomplete` treated separately; the remaining eight categories contain only complete CDS transcripts.

- **ensembldb (Ensembl 86)** and **GenomicFeatures (Ensembl 86)** matched fastCDS exactly (5,000/5,000 queries), with identical coordinates in every category.
- **TransVar (Ensembl 95)** reports a single genomic interval rather than individual CDS segments, so it can only be compared for queries that map to a single exon (single_exon_domain and single_exon_gene). Categories spanning multiple exons are therefore not applicable (NA).
- **Ensembl REST API (Ensembl 115)** achieved 99.06% agreement overall. All 37 coordinate differences (≤2 nt) occurred in the `cds_incomplete` category. Across the eight complete-CDS categories there were no coordinate or structural mismatches, although 10 queries could not be mapped by the REST service.

In [2]:
# Supplementary Table S1 - per-tool, per-category agreement with fastCDS
# (finalized 9-category run; built by benchmarks/make_table_s1.py).
table_s1 = pd.read_csv(REPO / "benchmarks" / "table_S1_agreement.tsv", sep="\t")
table_s1

,category,n,ensembldb_pct,GenomicFeatures_pct,TransVar_pct,REST_exact,REST_off,REST_no_map,REST_pct
0,single_exon_domain,1000,100,100,100.0,998,0,2,99.80
1,multi_exon_domain,1000,100,100,NaN,999,0,1,99.90
2,codon_split_boundary,500,100,100,NaN,498,0,2,99.60
3,plus_strand_gene,1000,100,100,NaN,997,0,3,99.70
4,minus_strand_gene,1000,100,100,NaN,1000,0,0,100.00
5,cds_incomplete,200,100,100,NaN,163,37,0,81.50
6,selenoprotein,100,100,100,NaN,98,0,2,98.00
7,single_exon_gene,100,100,100,100.0,100,0,0,100.00
8,many_exon_gene,100,100,100,NaN,100,0,0,100.00
9,OVERALL,5000,100,100,NaN,4953,37,10,99.06


### 1b. Where the Ensembl REST API differs

All coordinate differences between fastCDS and the Ensembl REST API occur in the `cds_incomplete` category, which contains transcripts annotated with `cds_start_NF` or `cds_end_NF`. Within this category, 163 of 200 queries match exactly, while the remaining 37 differ by no more than 2 nt at the truncated CDS boundary. No coordinate differences are observed in the eight complete-CDS categories, and both Bioconductor `proteinToGenome` implementations match fastCDS exactly.

## 2. Runtime and memory on the same human dataset

Every tool is run on the **same human v86 query set**, and
line up single-thread throughput and peak memory. fastCDS, ensembldb,
GenomicFeatures and the Ensembl REST API map human protein-domain queries
directly.

geneplot is a general tool (it takes any GFF + domain file), so it goes on the
human set too, with inputs built from the same Ensembl-86 annotation and Pfam
domains (`benchmarks/build_human_tool_inputs.py`, then
`benchmarks/matched/run_geneplot.py`). It has no index, and on the human genome that
bites: geneplot builds a `gffutils` SQLite database of the genome once (~3 min)
then walks each transcript per gene (**~8 genes/s**). On its tiny bundled example
(fruit-fly) it finishes in seconds; on real human data it runs in minutes, which
is why it is measured offline rather than inline. All numbers below are single
thread on the same machine and the same human set.

In [3]:
# Supplementary Table S2 - mapping speed and peak memory
# (finalized 5-tool run; built by benchmarks/make_table_s2.py).
table_s2 = pd.read_csv(REPO / "benchmarks" / "table_S2_speed_memory.tsv", sep="\t")
table_s2

,tool,queries_per_s,peak_rss_mb,n
0,fastCDS,5886.00,808,10000
1,GenomicFeatures (GRanges),27.50,1270,10000
2,geneplot,8.10,317,1000
3,ensembldb,6.00,1191,10000
4,Ensembl REST,1.29,28,1000


## Summary

- **fastCDS is orders of magnitude faster than every alternative on human data.**
  End-to-end it maps ~5,900 queries/s (N = 10,000). Of the tools benchmarked for
  speed in **Table S2**, the next-fastest is GenomicFeatures (GRanges) at ~28/s -
  so **fastCDS is more than 200-fold faster than GenomicFeatures** - then geneplot
  ~8/s, ensembldb ~6/s, and the Ensembl REST API ~1.3/s. (TransVar is a span-only
  variant-annotation tool, so it is compared for coordinate accuracy in Table S1,
  not in the speed table.)
- **The bundled-example speed is misleading.** geneplot looks quick on its small
  fruit-fly sample (seconds), but on the human genome it collapses into the
  slow-tool range, because it does not build a reusable index: it rebuilds a
  `gffutils` database and re-reads the domain file per gene. fastCDS pays the genome
  cost once (the index) and then maps in microseconds.
- **And fastCDS agrees with all of them.** On matched annotation (each tool on its
  native Ensembl release, same 9-category 5,000-query sample, Table S1) it is
  **100% exact to both Bioconductor `proteinToGenome` methods - ensembldb (v86) and
  GenomicFeatures (v86) - in every category**, **100% to TransVar (v95) on the
  single-CDS-block categories it can score** (span-only elsewhere), and **99.06% to
  Ensembl REST (v115)** - where every divergence is a `cds_start_NF` incomplete-CDS
  convention, not an error (see the exclusive `cds_incomplete` category, section 1b).
- **Memory:** fastCDS uses ~0.8 GB at N = 10,000 (mostly the loaded index); the
  Bioconductor tools sit at 1.2-1.3 GB (Table S2). Only
  fastCDS returns the per-CDS-exon decomposition; the others give the genomic
  envelope only.
- **Scale:** fastCDS runs to 1,000,000 queries (see [`scaling_and_ram.ipynb`](https://github.com/SotoLF/fastCDS/blob/main/reproduce_benchmarking_figures/notebooks/scaling_and_ram.ipynb)); the Bioconductor tools and geneplot cap at 10,000, and the rate-limited REST API at 1,000.
